# Notebook 10 — Executable Long-Only Multi-Lot Portfolio Backtest

This notebook executes the pre-registered Stage 10 portfolio rules. It consumes
the frozen Stage 09 signal stream and does not retrain the model or change the
signal policy.

Primary design:

- customer transaction-size-anchored capital proxy in IRR;
- long-only next-open execution;
- independent repeated-signal lots;
- 0.5% initial risk per lot;
- zero planned risk only after break-even-or-better trailing protection is
  executable;
- full fees, sell tax, slippage, cash, exposure, capacity, and liquidity;
- multi-lot vs single-lot and trailing vs fixed-target controls.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml


def resolve_repository_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (
            (candidate / "configs").exists()
            and (candidate / "src").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository root or its notebooks directory."
    )


REPOSITORY_ROOT = resolve_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.portfolio_backtest import (
    STAGE10_SCHEMA_VERSION,
    run_stage10,
)

CONFIG_PATH = REPOSITORY_ROOT / "configs" / "portfolio_backtest.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    stage10_config = yaml.safe_load(handle)

print("Repository root:", REPOSITORY_ROOT)
print("Configuration:", CONFIG_PATH)
print("Schema:", STAGE10_SCHEMA_VERSION)


In [ ]:
assert stage10_config["status"] == "stage_10_preregistered_v1"
assert stage10_config["market"]["direction"] == "long_only"
assert stage10_config["market"]["short_selling_allowed"] is False
assert stage10_config["execution"]["leverage_allowed"] is False
assert stage10_config["risk"]["risk_per_lot_fraction"] == 0.005
assert stage10_config["capacity"]["maximum_open_lots_per_symbol"] == 3
assert stage10_config["risk"]["protected_lot_risk_treatment"][
    "trailing_active_at_or_above_net_break_even"
] == "zero"
assert stage10_config["initial_capital"]["currency"] == "IRR"
assert stage10_config["initial_capital"][
    "expected_primary_initial_capital_irr"
] == 20316282773.64711

print("Stage 10 pre-registration assertions: PASSED")


## Customer input placement

Place the supplied `final_feature_dim.csv` in either:

- `data_ready/portfolio/final_feature_dim.csv` (preferred), or
- the repository root.

The notebook verifies 500 positive `avg_Buy` values and independently
recomputes the trimmed-mean capital proxy before any portfolio simulation.


In [ ]:
outputs = run_stage10(
    repository_root=REPOSITORY_ROOT,
    config=stage10_config,
    write_outputs=True,
)

print("\nNotebook 10 internal integrity checks: PASSED")


In [ ]:
capital = outputs["capital_summary"]
print("Currency:", capital["currency"])
print(
    "Primary initial capital:",
    f'{capital["primary_initial_capital_irr"]:,.2f}',
    "IRR",
)
print(
    "Trimmed mean avg_Buy:",
    f'{capital["trimmed_mean_avg_buy_irr"]:,.2f}',
    "IRR",
)
print(
    "Cost-adjusted stop-loss fraction:",
    f'{capital["cost_adjusted_stop_loss_fraction"]:.12%}',
)
print(
    "Implied initial lot weight:",
    f'{capital["implied_initial_lot_weight"]:.12%}',
)


In [ ]:
summary = outputs["scenario_summary"].copy()
primary = summary.loc[summary["is_primary_scenario"]].iloc[0]

primary_fields = [
    "scenario_id",
    "initial_capital_irr",
    "final_equity_irr",
    "total_return",
    "cagr",
    "maximum_drawdown",
    "sharpe_zero_risk_free",
    "sortino_zero_target",
    "accepted_signals",
    "rejected_signals",
    "closed_trades",
    "net_win_rate",
    "net_payoff_ratio",
    "net_profit_factor",
    "maximum_open_lots",
    "maximum_open_symbols",
    "maximum_planned_open_risk_fraction",
    "maximum_gross_exposure_fraction",
    "trailing_activated_trades",
]

print("\nPrimary Stage 10 scenario")
for field in primary_fields:
    print(f"{field}: {primary[field]}")


In [ ]:
integrity = outputs["integrity_audit"]

assert integrity["entry_constraint_violations"].eq(0).all()
assert integrity["negative_cash_events"].eq(0).all()
assert integrity[
    "protected_lot_nonzero_planned_risk_events"
].eq(0).all()
assert integrity["open_lots_after_tail"].eq(0).all()
assert integrity["all_positions_long_only"].all()
assert (
    integrity["accepted_signals"] == integrity["closed_trades"]
).all()

print("\nPortfolio integrity audit: PASSED")
print("Scenarios executed:", len(integrity))
print("All positions long-only:", True)
print("Protected-lot planned-risk violations:", 0)
print("Negative-cash events:", 0)
print("Open lots after tail:", 0)


In [ ]:
decision_summary = (
    outputs["signal_decisions"]
    .groupby(
        ["scenario_id", "decision", "rejection_reason"],
        dropna=False,
        sort=True,
    )
    .size()
    .rename("events")
    .reset_index()
)

exit_summary = (
    outputs["trade_ledger"]
    .groupby(["scenario_id", "exit_reason"], sort=True)
    .size()
    .rename("trades")
    .reset_index()
)

print("\nTop primary rejection reasons")
display(
    decision_summary.loc[
        (decision_summary["scenario_id"] == primary["scenario_id"])
        & (decision_summary["decision"] == "rejected")
    ]
    .sort_values("events", ascending=False)
    .head(15)
)

print("\nPrimary exit reasons")
display(
    exit_summary.loc[
        exit_summary["scenario_id"] == primary["scenario_id"]
    ].sort_values("trades", ascending=False)
)


In [ ]:
manifest_path = (
    REPOSITORY_ROOT
    / "results"
    / "manifests"
    / "10_portfolio_backtest_manifest.json"
)
with manifest_path.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print("\nStage 10 manifest:", manifest_path)
print("Git commit:", manifest["git_commit_sha"])
print("Configuration hash:", manifest["configuration_hash"])
print(
    "Stage 09 inference lock:",
    manifest["stage09_inference_lock_sha256"],
)
print(
    "Primary scenario:",
    manifest["primary_scenario"]["scenario_id"],
)
print(
    "\nNext action: independently audit Stage 10 outputs before freezing "
    "the portfolio stage."
)
